**Step 1: Import the libraries**



In [ ]:
import pandas as pd
import requests
import sqlite3
import matplotlib.pyplot as plt

**Step 2: Collect data from the GitHub API**

In [ ]:
url =  "https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc&per_page=100"

In [ ]:
response = requests.get(url)

In [ ]:
data = response.json()

In [ ]:
print("Data collected successfully!")

In [ ]:
print("Number of repositories:", len(data["items"]))

**Step 3: Create the DataFrame**

In [ ]:
df = pd.DataFrame(data["items"])

In [ ]:
df.head()

In [ ]:
print("Rows and columns:", df.shape)

In [ ]:
df.info()

In [ ]:
df. columns

**Step 4: Select the required columns**

In [ ]:
df = df[
    [
        "name",
        "owner",
        "language",
        "stargazers_count",
        "forks_count",
        "watchers_count",
        "open_issues_count",
        "created_at",
        "updated_at",
        "license"
    ]
]

In [ ]:
df.head()

**Step 5: Extract the owner username**

In [ ]:
def get_owner_name(owner):
    if owner:
        return owner["login"]
    else:
        return "Unknown"

In [ ]:
df["owner"] = df["owner"].apply(get_owner_name)

In [ ]:
df[["name", "owner"]].head()

**Step 6: Extract the license name**

In [ ]:
def get_license_name(license):
    if license:
        return license["name"]
    else:
        return "Unknown"

In [ ]:
df["license"] = df["license"].apply(get_license_name)

In [ ]:
df[["name", "license"]].head()

**Step 7: Check missing values**

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
df["language"] = df["language"].fillna

In [ ]:
print("Missing values after cleaning:")
print(df.isnull().sum())

**Step 8: Check for duplicate records**

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Number of duplicate rows after cleaning:", df.duplicated().sum())

**Step 9: Convert the date columns**

In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"])

In [ ]:
df["updated_at"] = pd.to_datetime(df["updated_at"])

**Step 10: Rename the columns**

In [ ]:
df = df.rename(
    columns={
        "stargazers_count": "stars",
        "forks_count": "forks",
        "watchers_count": "watchers",
        "open_issues_count": "open_issues",
        "created_at": "created_date",
        "updated_at": "updated_date"
    }
)

**Step 11: Check the cleaned dataset**

In [ ]:
df.head()

In [ ]:
print(df.columns)

In [ ]:
df.info()

In [ ]:
print("Final number of repositories:", len(df))

**Step 12: Save the dataset as CSV**

In [ ]:
df.to_csv("github_projects.csv", index=False)
print("github_projects.csv has been saved successfully!")

In [ ]:
connection = sqlite3.connect("github_projects.db")
print("Database created successfully!")

In [ ]:
df = pd.read_csv("github_projects.csv")
df.head()

In [ ]:
df.to_sql("Repositories", connection, if_exists="replace", index=False)

print("Data has been added to the Repositories table.")

In [ ]:
query = "SELECT * FROM Repositories"

repositories = pd.read_sql(query, connection)

repositories.head()

**Step 13: Repositories with more than 10,000 stars**

In [ ]:
query = """
SELECT *
FROM Repositories
WHERE stars > 10000
"""

result = pd.read_sql(query, connection)

result

**Step 14: Repository names containing "Machine"**

In [ ]:
query = """
SELECT *
FROM Repositories
WHERE name LIKE '%Machine%'
"""

result = pd.read_sql(query, connection)

result

**Step 15:  Logical Operators(OR, AND, NOT) **

In [ ]:
query = """
SELECT *
FROM Repositories
WHERE stars > 10000
AND forks > 1000
"""

result = pd.read_sql(query, connection)

result

In [ ]:
query = """
SELECT *
FROM Repositories
WHERE stars > 10000
OR forks > 5000
"""

result = pd.read_sql(query, connection)

result

In [ ]:
query = """
SELECT *
FROM Repositories
WHERE NOT language = 'Unknown'
"""

result = pd.read_sql(query, connection)

result

**Step 16 :Sorting and Finding the Top 10**

In [ ]:
query = """
SELECT *
FROM Repositories
ORDER BY stars DESC
"""

result = pd.read_sql(query, connection)

result.head()

In [ ]:
query = """
SELECT name, owner, language, stars
FROM Repositories
ORDER BY stars DESC
LIMIT 10
"""

top_10 = pd.read_sql(query, connection)

top_10

**Step 17: Calculate Some Basic Statistics**

In [ ]:
query = """
SELECT COUNT(*) AS total_repositories
FROM Repositories
"""

total_repositories = pd.read_sql(query, connection)

total_repositories

In [ ]:
query = """
SELECT AVG(stars) AS average_stars
FROM Repositories
"""

average_stars = pd.read_sql(query, connection)

average_stars

In [ ]:
query = """
SELECT
    COUNT(*) AS total_repositories,
    AVG(stars) AS average_stars
FROM Repositories
"""

summary = pd.read_sql(query, connection)

summary

**Step 18: Analyze Programming Languages**

In [ ]:
query = """
SELECT
    language,
    COUNT(*) AS repository_count
FROM Repositories
GROUP BY language
HAVING COUNT(*) > 5
ORDER BY repository_count DESC
"""

language_analysis = pd.read_sql(query, connection)

language_analysis

**Step 19:Create Visualizations**

Chart 1: Top 10 Most Popular Repositories

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(top_10["name"], top_10["stars"])

plt.xlabel("Stars")
plt.ylabel("Repository")
plt.title("Top 10 Most Popular GitHub Repositories")

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(top_10["name"], top_10["stars"])

plt.xlabel("Repository")
plt.ylabel("Stars")
plt.title("Top 10 Most Popular GitHub Repositories")

plt.tight_layout()

plt.show()

Chart 2: Repository Creation Trends

In [ ]:
query = """
SELECT created_date
FROM Repositories
"""

creation_dates = pd.read_sql(query, connection)

In [ ]:
creation_dates["created_date"] = pd.to_datetime(
    creation_dates["created_date"]
)

In [ ]:
creation_dates["year"] = creation_dates["created_date"].dt.year

In [ ]:
yearly_repositories = (
    creation_dates
    .groupby("year")
    .size()
    .reset_index(name="repository_count")
)

yearly_repositories

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    yearly_repositories["year"],
    yearly_repositories["repository_count"],
    marker="o"
)

plt.xlabel("Year")
plt.ylabel("Number of Repositories")
plt.title("Repository Creation Trends Over Time")

plt.show()